# PZ_Proyecto_integrador

Todos los datos son **simulados**. Lee primero las tres notas del día. Completa una celda y ejecuta su comprobación; esta fallará hasta resolver el ejercicio. Resolver aquí cuenta como los ejercicios de la nota, no duplica el trabajo.

La primera celda de código contiene los generadores pertinentes de P.D, idénticos a la nota. P1–P3: solo serie_anual. P4: añade serie_mensual. P5: añade malla. Las funciones importan dependencias al llamarse. Si faltan bibliotecas ejecuta antes la instalación opcional.

Ejecuta en un directorio de práctica: se crean CSV y NetCDF allí. No se necesitan archivos de datos externos.

In [ ]:
def serie_anual(n=24, tasa=0.02, nivel=14.0):
    """Serie simulada sin ruido; un valor por año desde 2000, en °C."""
    valores = []  # Lista vacía: iremos añadiendo valores.
    for i in range(n):
        valores.append(nivel + tasa * i)
    return valores

def serie_mensual(semilla=2026, faltantes=False):
    """Tres regiones simuladas, mensuales, 2000–2023; jamás datos NASA."""
    import numpy as np
    import pandas as pd
    rng = np.random.default_rng(semilla)
    fechas = pd.date_range("2000-01-01", periods=288, freq="MS")
    t = np.arange(288) / 12  # Tiempo en años desde el inicio.
    tablas = []
    for region, tasa in zip(["Costa", "Sierra", "Llanura"], [0.02, 0.04, -0.01]):
        estacion = 2 * np.sin(2 * np.pi * t)
        variacion = 0.12 * np.sin(2 * np.pi * t / 5)
        ruido = rng.normal(0, 0.08, len(t))
        valor = 14 + tasa * t + estacion + variacion + ruido
        if faltantes:
            valor[150::53] = np.nan  # Base 2000–2009 completa.
        tablas.append(pd.DataFrame({"fecha": fechas, "region": region,
                                   "temp_C": valor}))
    return pd.concat(tablas, ignore_index=True)

def malla(semilla=2026):
    """Dataset simulado mensual: time=24, lat=3, lon=4; temperatura en °C."""
    import numpy as np
    import pandas as pd
    import xarray as xr
    rng = np.random.default_rng(semilla)
    tiempo = pd.date_range("2000-01-01", periods=24, freq="MS")
    lat = np.array([-30.0, 0.0, 30.0])
    lon = np.array([-120.0, -90.0, -60.0, -30.0])
    t = np.arange(24).reshape(24, 1, 1) / 12
    tasas = np.array([[-0.03, -0.01, 0.01, 0.03]] * 3)
    valor = 14 + t * tasas + 0.3 * np.sin(2 * np.pi * t)
    valor = valor + rng.normal(0, 0.01, (24, 3, 4))
    ds = xr.Dataset({"temp": (("time", "lat", "lon"), valor)},
                    coords={"time": tiempo, "lat": lat, "lon": lon})
    ds.attrs["origen"] = "simulado"
    ds["temp"].attrs["units"] = "degC"
    ds["lat"].attrs["units"] = "degrees_north"
    ds["lon"].attrs["units"] = "degrees_east"
    return ds

## Instalación opcional para Colab

%pip instala en el entorno del notebook. Ejecuta una vez antes de usar bibliotecas; reinicia si cambiaste versiones. En local usa venv y requirements.txt.

In [ ]:
%pip install -q "numpy>=2,<3" "pandas>=2.2,<3" matplotlib scipy xarray netCDF4 statsmodels pymannkendall pytest

Las etapas comparten variables: ejecuta en orden. Git solo se inicializa en una carpeta de práctica; **no se ejecuta commit ni push**.

## Núcleo · 95 min

### PZ-01 · ⭐⭐⭐ · Reto Tierra · 15 min · Generar, leer y limpiar

**Datos simulados.** Genera las tres regiones de P.D con faltantes. Pasa por un CSV en memoria, convierte fechas, ordena y cuenta descartes. Comprueba unicidad fecha/región; duplicated marca filas repetidas según esas columnas. Conserva df y limpio.

<details><summary>Pista</summary>

No rellenes ausencias; los conteos anuales detectarán meses excluidos.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(df)==864 and descartes==9 and len(limpio)==855), "Generar, leer y limpiar: revisa la etapa y sus entradas"

### PZ-02 · ⭐⭐⭐ · Reto Tierra · 20 min · Climatología y cobertura anual

**Datos simulados.** Por región calcula referencias mensuales exclusivamente en 2000–2009. Guarda series_anuales con anomalías anuales y cobertura con conteos de meses. Exige doce meses; usa índices de fechas reales.

<details><summary>Pista</summary>

Los años 2012, 2016 y 2021 están incompletos y deben quedar ausentes.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(series_anuales)==3 and all([s.notna().sum()==21 for s in series_anuales.values()])), "Climatología y cobertura anual: revisa la etapa y sus entradas"

### PZ-03 · ⭐⭐⭐ · Reto Tierra · 20 min · Estimar y exportar tasas

**Datos simulados.** Ajusta linregress por región con años reales, centrados en 2000. Guarda ajustes y tabla con tasa_decada,p_ols,n_anios. Incluye periodo,base,metodo,unidad,origen y exporta resultados_simulados.csv.

<details><summary>Pista</summary>

No confundas las 21 observaciones válidas con 21 años consecutivos.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (tabla["region"].to_list()==["Sierra","Costa","Llanura"] and (tabla["n_anios"]==21).all() and np.allclose(tabla["tasa_decada"],[0.4,0.2,-0.1],atol=0.04)), "Estimar y exportar tasas: revisa la etapa y sus entradas"

### PZ-04 · ⭐⭐⭐ · Reto Tierra · 25 min · Figura de evidencia

**Datos simulados.** Dibuja series anuales con huecos visibles y sus rectas, más barras de tasas. Conserva fig,ejes. En el notebook basta ver la figura; el archivo incluido se regenera con assets/generar_figuras.py.

<details><summary>Pista</summary>

axhline traza una referencia horizontal y tight_layout ajusta márgenes. Se explican aquí antes de usarlos.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(ejes[0].lines)==6 and len(ejes[1].patches)==3), "Figura de evidencia: revisa la etapa y sus entradas"

### PZ-05 · ⭐⭐⭐ · Reto Tierra · 15 min · Conclusión de cinco líneas

**Datos simulados.** Escribe conclusion con cinco líneas: pregunta, datos, método, resultado y límite. Incluye la tasa estimada de Sierra, unidad, 21 años válidos y advertencia de datos simulados. splitlines separa un texto en líneas.

<details><summary>Pista</summary>

La tasa y la cobertura pertenecen al resultado; la validez estadística necesita los bloques B y C.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(conclusion.splitlines())==5 and "simulad" in conclusion and "°C/década" in conclusion), "Conclusión de cinco líneas: revisa la etapa y sus entradas"

## Ampliación · 120 min

### PZ-06 · ⭐⭐⭐ · Reto Tierra · 20 min · Estimación robusta

**Datos simulados.** Calcula theilslopes(y,x) para cada región con años reales. Guarda robustas con tasa_decada_theil, inferior y superior por década.

<details><summary>Pista</summary>

Los límites son para la pendiente bajo los supuestos del estimador.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(robustas)==3 and (robustas["inferior"]<=robustas["tasa_decada_theil"]).all() and (robustas["superior"]>=robustas["tasa_decada_theil"]).all()), "Estimación robusta: revisa la etapa y sus entradas"

### PZ-07 · ⭐⭐⭐ · Reto Tierra · 20 min · Comparar métodos

**Datos simulados.** Une tabla y robustas por región; calcula diferencia entre tasas. Identifica el mayor desacuerdo absoluto en max_diferencia.

<details><summary>Pista</summary>

one_to_one exige claves únicas en ambas tablas. Acuerdo entre métodos no prueba ausencia de sesgo.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (len(comparacion)==3 and max_diferencia<0.05), "Comparar métodos: revisa la etapa y sus entradas"

### PZ-08 · ⭐⭐⭐ · Reto Tierra · 20 min · Tasa por celda

**Datos simulados.** Genera la malla P.D. Reduce estacionalidad con promedios anuales antes de estimar pendiente por celda. Solo hay dos años: es una demostración de programación, sin prueba de significancia. Conserva tasa_mapa como DataArray.

<details><summary>Pista</summary>

Cada celda recibe los mismos años; no interpretes pvalue con esta cobertura mínima.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (tasa_mapa.shape==(3,4) and bool((tasa_mapa.isel(lon=0)<0).all()) and bool((tasa_mapa.isel(lon=-1)>0).all())), "Tasa por celda: revisa la etapa y sus entradas"

### PZ-09 · ⭐⭐⭐ · Reto Tierra · 20 min · Mapa y metadatos

**Datos simulados.** Dibuja tasa_mapa con límites simétricos y guarda un Dataset NetCDF con el campo tasa. Conserva artista y archivo_mapa.

<details><summary>Pista</summary>

to_dataset asigna nombre a una variable para exportarla; cbar_kwargs configura la barra de color.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (artista.get_clim()==(-limite,limite) and archivo_mapa["tasa"].attrs["origen"]=="simulado"), "Mapa y metadatos: revisa la etapa y sus entradas"

### PZ-10 · ⭐⭐⭐ · Reto Tierra · 20 min · README y repositorio local

**Datos simulados.** Crea repo_simulado con README.md y .gitignore. Inicializa Git local con git init y consulta status mediante subprocess.run, que ejecuta una lista de argumentos externos; check=True detecta fallo. No ejecutes commit ni push. shutil.which detecta si Git está disponible; si falta, conserva los documentos.

<details><summary>Pista</summary>

Este repositorio de práctica no contiene commits. Si falta Git, la rúbrica deja la inicialización pendiente.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert ((repo/"README.md").exists() and (repo/".gitignore").exists() and (not git_disponible or (repo/".git").is_dir())), "README y repositorio local: revisa la etapa y sus entradas"

### PZ-11 · ⭐⭐⭐ · Reto Tierra · 20 min · Entrega reproducible

**Datos simulados.** Repite el generador con la misma semilla, verifica igualdad y agrega al README las versiones reales. importlib.metadata.version consulta la versión de una distribución instalada. Deja plan_git como texto con add y commit para lectura, sin ejecutarlo.

<details><summary>Pista</summary>

El plan de comandos se guarda como texto; esta etapa no crea commits ni publica nada.

</details>

In [ ]:
# TU CÓDIGO AQUÍ

In [ ]:
assert (replica.equals(original) and "Versiones verificadas" in (repo/"README.md").read_text(encoding="utf-8")), "Entrega reproducible: revisa la etapa y sus entradas"